# CodeCraft — Early Metabolic Risk Detection
## Notebook 01: Dataset Loading, Cleaning, Merging, Target Construction, and Validation

**Author:** Senior Data Engineer & ML Researcher  
**Project:** CodeCraft (Early Metabolic Risk Detection using NHANES 2017–2018)  
**Phase:** Data Engineering & Pipeline Foundation

---

## Section 1 — Project Introduction

### Project Objective & Scope
The **CodeCraft** project aims to build a rigorous machine learning, deep learning, and quantum machine learning (QML) pipeline for early metabolic risk detection. By analyzing biomarker and demographic data from the National Health and Nutrition Examination Survey (NHANES) 2017–2018 cycle, we identify HbA1c-defined prediabetes / elevated metabolic risk categories among adults.

### Why NHANES 2017–2018?
The National Health and Nutrition Examination Survey (NHANES) is a program of studies designed to assess the health and nutritional status of adults and children in the United States. The 2017–2018 cycle provides standardized, high-quality laboratory measurements (including glycohemoglobin), standardized physical examinations, and detailed demographic interviews.

### Source Files Used
1. **`DEMO_J.XPT`**: Demographic variables (Age, Gender, Race/Ethnicity, Survey Weights, etc.).
2. **`BMX_J.XPT`**: Body measurements (BMI, Waist Circumference, Weight, Height, etc.).
3. **`GHB_J.XPT`**: Glycohemoglobin (HbA1c) laboratory measurements.

### Workflow Overview
1. **Path Detection & Validation:** Locate source `.XPT` files securely across local storage or Google Colab.
2. **Loading:** Read SAS transport files into pandas DataFrames.
3. **Schema Validation:** Verify columns, participant IDs (`SEQN`), and data types against official CDC documentation.
4. **Merging:** Join datasets on participant identifier (`SEQN`) using strict inner joins, tracking participant flow.
5. **Population Filtering:** Restrict study population to adults aged $\ge 18$ years.
6. **Target Construction:** Define `risk_label` based on HbA1c (`LBXGH`): Normal ($<5.7\%$) vs. Elevated Risk ($5.7\% \le \text{HbA1c} < 6.5\%$), excluding diabetes ($\ge 6.5\%$) and missing values.
7. **Leakage Prevention:** Isolate and remove target-leaking features (`LBXGH`, self-reported diabetes, etc.) from the predictor matrix.
8. **Quality Analysis:** Perform missingness analysis, distribution checks, and data sanity audits.
9. **Survey Design Preservation:** Preserve NHANES sampling weights (`WTINT2YR`, `WTMEC2YR`) and design variables (`SDMVPSU`, `SDMVSTRA`) for downstream survey-weighted analysis.
10. **Export:** Save clean processed datasets (`.csv`) and JSON audit reports.
11. **Final Verification:** Execute rigorous programmatic assertions.

### Methodological Principles & Clinical Disclaimer
- **Reproducibility:** A fixed random seed (`42`) is set across all stochastic operations.
- **Data Leakage Prevention:** Strict separation of targets and audit variables from input features.
- **Clinical Disclaimer:** *This project identifies HbA1c-defined risk categories in cross-sectional survey data for research and modeling demonstrations. It does not establish a clinical diagnosis or predict future disease onset.*

## Section 2 — Install and Import Dependencies

We require standard data engineering and machine learning libraries:
- `pandas`: Data manipulation and tabular analysis
- `numpy`: Numerical operations
- `pyreadstat`: Fast reader for SAS transport files (`.XPT`)
- `matplotlib` & `seaborn`: Exploratory visualization
- `scikit-learn`: Metric utilities and preprocessing foundations (no model training in this notebook)

Let's check and install required packages if running in Google Colab or environments missing `pyreadstat`.

In [ ]:
# Install pyreadstat if not already installed (useful for Google Colab)
try:
    import pyreadstat
except ImportError:
    print("Installing pyreadstat for reading SAS XPT files...")
    !pip install -q pyreadstat

import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Configure display settings
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
sns.set_theme(style='whitegrid')

print("All libraries imported successfully.")

## Section 3 — Configure Dataset Paths

We define a robust path detection mechanism supporting:
1. Local project `raw/` folder (`PROJECT/raw/`)
2. Alternate local `data/raw/` folder
3. Google Colab environment (`/content/raw/` or uploaded files)
4. Custom user-configured paths

We verify that each expected file (`DEMO_J.XPT`, `BMX_J.XPT`, `GHB_J.XPT`) exists and has a non-zero file size.

In [ ]:
def find_raw_data_dir():
    """Search for raw NHANES .XPT files across common candidate directories."""
    candidate_paths = [
        Path("raw"),
        Path("data/raw"),
        Path("../raw"),
        Path("../data/raw"),
        Path("/content/raw"),
        Path("."),
    ]
    
    required_files = ["DEMO_J.XPT", "BMX_J.XPT", "GHB_J.XPT"]
    
    for basePath in candidate_paths:
        if basePath.exists() and basePath.is_dir():
            if all((basePath / fname).exists() for fname in required_files):
                return basePath.resolve()
                
    # Check if files are in current working directory directly
    if all(Path(fname).exists() for fname in required_files):
        return Path(".").resolve()
        
    raise FileNotFoundError(
        f"Could not locate all required NHANES files ({required_files}) in candidate paths: {candidate_paths}. "
        "Please place DEMO_J.XPT, BMX_J.XPT, and GHB_J.XPT into the 'raw/' directory."
    )

RAW_DATA_DIR = find_raw_data_dir()
print(f"[INFO] Detected raw data directory: {RAW_DATA_DIR}")

DEMO_PATH = RAW_DATA_DIR / "DEMO_J.XPT"
BMX_PATH = RAW_DATA_DIR / "BMX_J.XPT"
GHB_PATH = RAW_DATA_DIR / "GHB_J.XPT"

file_paths = {
    "DEMO_J.XPT": DEMO_PATH,
    "BMX_J.XPT": BMX_PATH,
    "GHB_J.XPT": GHB_PATH
}

file_info = []
for name, path in file_paths.items():
    if not path.exists():
        raise FileNotFoundError(f"Required file {name} not found at {path}")
    size_bytes = path.stat().st_size
    if size_bytes == 0:
        raise ValueError(f"File {name} at {path} is empty (0 bytes).")
    file_info.append({"File": name, "Path": str(path), "Size (MB)": round(size_bytes / (1024 * 1024), 2)})

df_file_info = pd.DataFrame(file_info)
display(df_file_info)

## Section 4 — Load the Original NHANES Files

We load the SAS transport files using `pandas.read_sas` (with `pyreadstat` engine backing).  
We inspect:
- Dataset dimensions (rows $\times$ columns)
- Column names and data types
- First 5 rows
- Missing-value counts
- Participant ID (`SEQN`) duplicate checks

We assign descriptive variable names: `demographics_df`, `body_measurements_df`, and `glycohemoglobin_df`.

In [ ]:
# Load datasets
demographics_df = pd.read_sas(DEMO_PATH, format="xport")
body_measurements_df = pd.read_sas(BMX_PATH, format="xport")
glycohemoglobin_df = pd.read_sas(GHB_PATH, format="xport")

print(f"Loaded demographics_df: {demographics_df.shape}")
print(f"Loaded body_measurements_df: {body_measurements_df.shape}")
print(f"Loaded glycohemoglobin_df: {glycohemoglobin_df.shape}")

In [ ]:
def inspect_dataset(df, name):
    print(f"=== {name} ===")
    print(f"Shape: {df.shape}")
    duplicate_seqns = df['SEQN'].duplicated().sum()
    print(f"Duplicate SEQN count: {duplicate_seqns}")
    print(f"Missing values total: {df.isna().sum().sum()}")
    print("First 5 rows:")
    display(df.head())
    print("\n")

inspect_dataset(demographics_df, "Demographics (DEMO_J)")
inspect_dataset(body_measurements_df, "Body Measurements (BMX_J)")
inspect_dataset(glycohemoglobin_df, "Glycohemoglobin (GHB_J)")

## Section 5 — Understand and Validate the Schema

We reference official CDC 2017–2018 documentation:
- DEMO: https://wwwn.cdc.gov/Nchs/Nhanes/2017-2018/DEMO_J.htm
- BMX: https://wwwn.cdc.gov/Nchs/Nhanes/2017-2018/BMX_J.htm
- GHB: https://wwwn.cdc.gov/Nchs/Nhanes/2017-2018/GHB_J.htm

Critical expected columns:
- `SEQN`: Respondent sequence number (participant ID)
- `RIDAGEYR`: Age in years at screening
- `RIAGENDR`: Gender (1 = Male, 2 = Female)
- `BMXBMI`: Body Mass Index ($kg/m^2$)
- `BMXWAIST`: Waist Circumference ($cm$)
- `LBXGH`: Glycohemoglobin / HbA1c ($\%$)
- Survey weight & design variables: `WTINT2YR`, `WTMEC2YR`, `SDMVPSU`, `SDMVSTRA`

In [ ]:
required_columns = {
    'demographics_df': ['SEQN', 'RIDAGEYR', 'RIAGENDR', 'WTINT2YR', 'WTMEC2YR', 'SDMVPSU', 'SDMVSTRA'],
    'body_measurements_df': ['SEQN', 'BMXBMI', 'BMXWAIST'],
    'glycohemoglobin_df': ['SEQN', 'LBXGH']
}

for df_name, cols in required_columns.items():
    df = locals()[df_name]
    missing_cols = [c for c in cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"{df_name} is missing required columns: {missing_cols}")
    print(f"[VALIDATED] {df_name} contains all required columns: {cols}")

## Section 6 — Merge the Three Datasets

We merge the three datasets on participant ID (`SEQN`).  
Because participants may participate in interviews (`DEMO`) but not physical examinations (`BMX`) or laboratory tests (`GHB`), we use an inner join strategy to ensure complete feature and target availability.

In [ ]:
print(f"Original rows — Demographics: {len(demographics_df)}")
print(f"Original rows — Body Measurements: {len(body_measurements_df)}")
print(f"Original rows — Glycohemoglobin: {len(glycohemoglobin_df)}")

# Merge Demographics with Body Measurements
merged_1 = pd.merge(demographics_df, body_measurements_df, on='SEQN', how='inner')
print(f"After merging Demo & BMX: {len(merged_1)} rows (Lost: {len(demographics_df) - len(merged_1)})")

# Merge with Glycohemoglobin
merged_full = pd.merge(merged_1, glycohemoglobin_df, on='SEQN', how='inner')
print(f"After merging with GHB (Final raw merge): {len(merged_full)} rows (Lost from merged_1: {len(merged_1) - len(merged_full)})")

# Verify SEQN uniqueness
assert merged_full['SEQN'].is_unique, "SEQN is not unique in the merged dataset!"
print(f"[VERIFIED] Final merged dataset has {len(merged_full)} unique participants (one row per SEQN).")

## Section 7 — Create the Adult Study Population

NHANES includes participants of all ages. For metabolic risk detection (prediabetes/type 2 diabetes risk), we restrict the study population to adults aged 18 years and older (`RIDAGEYR >= 18`).

In [ ]:
rows_before_age = len(merged_full)

# Handle missing age explicitly
missing_age_count = merged_full['RIDAGEYR'].isna().sum()
print(f"Missing age values: {missing_age_count}")

# Filter for adults (>= 18)
adults_df = merged_full[merged_full['RIDAGEYR'] >= 18].copy()
rows_after_age = len(adults_df)

print(f"Rows before age filter: {rows_before_age}")
print(f"Rows after age filter (Age >= 18): {rows_after_age}")
print(f"Excluded minors (< 18) or missing age: {rows_before_age - rows_after_age}")

## Section 8 — Construct the Target Variable

We construct the target variable `risk_label` using glycohemoglobin (`LBXGH`) according to standard clinical thresholds:
- **Normal:** $\text{LBXGH} < 5.7\%$
- **Elevated Risk (Prediabetes):** $5.7\% \le \text{LBXGH} < 6.5\%$
- **Excluded (Diabetes / Diagnosis Range):** $\text{LBXGH} \ge 6.5\%$
- **Excluded:** Missing `LBXGH` values.

### Boundary Assertions & Testing
We explicitly test boundary values to ensure correct categorization.

In [ ]:
def assign_risk_label(hba1c):
    if pd.isna(hba1c):
        return np.nan
    elif hba1c < 5.7:
        return 'Normal'
    elif 5.7 <= hba1c < 6.5:
        return 'Elevated Risk'
    else:
        return 'Diabetes / Excluded'

# Test boundary values function
assert assign_risk_label(5.69) == 'Normal'
assert assign_risk_label(5.70) == 'Elevated Risk'
assert assign_risk_label(6.49) == 'Elevated Risk'
assert assign_risk_label(6.50) == 'Diabetes / Excluded'
assert pd.isna(assign_risk_label(np.nan))
print("[PASSED] Boundary assertions verified successfully.")

In [ ]:
total_adults_before_target = len(adults_df)

# Check missing HbA1c
missing_ghb = adults_df['LBXGH'].isna().sum()

# Apply target labeling
adults_df['raw_risk_label'] = adults_df['LBXGH'].apply(assign_risk_label)

# Count diabetes range exclusions
diabetes_count = (adults_df['raw_risk_label'] == 'Diabetes / Excluded').sum()

# Filter analysis dataset (Normal and Elevated Risk only)
analysis_df = adults_df[adults_df['raw_risk_label'].isin(['Normal', 'Elevated Risk'])].copy()
analysis_df = analysis_df.rename(columns={'raw_risk_label': 'risk_label'})

print(f"Total adult participants before target filtering: {total_adults_before_target}")
print(f"Excluded due to missing HbA1c: {missing_ghb}")
print(f"Excluded due to diabetes range (HbA1c >= 6.5%): {diabetes_count}")
print(f"Final analysis population size: {len(analysis_df)}")

print("\nTarget Class Distribution in Final Analysis Population:")
class_counts = analysis_df['risk_label'].value_counts()
class_proportions = analysis_df['risk_label'].value_counts(normalize=True) * 100
dist_df = pd.DataFrame({'Count': class_counts, 'Percentage (%)': class_proportions.round(2)})
display(dist_df)

## Section 9 — Prevent Target Leakage

To prevent data leakage, we must strictly ensure that variables directly derived from HbA1c, or diagnostic indicators of diabetes, are not included in the feature set ($X$).

**Excluded Columns:**
- `SEQN` (Participant identifier — not a predictive feature)
- `LBXGH` (Glycohemoglobin percentage — the exact basis of the target variable)
- Any transformation, binning, or derivative of `LBXGH`
- Self-reported diabetes diagnosis or medication variables (if present in survey responses)

We preserve `LBXGH` in an audit table for verification, but strip it from the model-ready feature matrix.

In [ ]:
leakage_columns = ['SEQN', 'LBXGH']

# Identify any other potential columns containing 'GH' or 'GLYCO' or 'DIABETES'
potential_leakage = [c for c in analysis_df.columns if any(term in c.upper() for term in ['GH', 'GLYCO', 'DIAB']) and c != 'risk_label']
print(f"Potential leakage columns identified: {potential_leakage}")

all_leakage_to_remove = list(set(leakage_columns + potential_leakage))

# Create audit table (retaining SEQN, LBXGH, risk_label)
audit_df = analysis_df[['SEQN', 'LBXGH', 'risk_label']].copy()

# Create model-ready feature matrix (dropping leakage columns)
model_ready_df = analysis_df.drop(columns=[c for c in all_leakage_to_remove if c in analysis_df.columns])

print(f"Model-ready dataset shape: {model_ready_df.shape}")
print(f"Predictor columns count: {model_ready_df.shape[1] - 1} (excluding target 'risk_label')")

## Section 10 — Missing Data and Data Quality Analysis

We perform a comprehensive data quality check:
- Missing counts and percentages per column
- Numeric summary statistics (Age, BMI, Waist circumference)
- Distribution checks and anomalies
- Participant uniqueness checks

In [ ]:
# Missing values report
missing_counts = model_ready_df.isna().sum()
missing_pct = (model_ready_df.isna().mean() * 100).round(2)
missing_report = pd.DataFrame({'Missing Count': missing_counts, 'Missing Percentage (%)': missing_pct})
missing_report = missing_report[missing_report['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)

print(f"Columns with missing values: {len(missing_report)}")
if len(missing_report) > 0:
    display(missing_report.head(20))
else:
    print("No missing values found in model-ready features.")

In [ ]:
# Numeric summary statistics for key variables
key_numeric_cols = [c for c in ['RIDAGEYR', 'BMXBMI', 'BMXWAIST'] if c in model_ready_df.columns]
display(model_ready_df[key_numeric_cols].describe())

## Section 11 — Survey Weights and Study Limitations

NHANES employs a complex, multistage probability sampling design. To make nationally representative inferences, survey weights (`WTINT2YR`, `WTMEC2YR`) and design variables (`SDMVPSU`, `SDMVSTRA`) must be accounted for.

- **Preservation:** We retain survey weights and PSU/strata variables in our processed datasets.
- **Modeling Rule:** Survey weights are **never** used as ordinary predictor features in machine learning models to prevent biasing algorithm feature attribution. Instead, they are utilized during survey-weighted evaluation or post-stratification analyses.
- **Limitations:** Cross-sectional analyses establish associations rather than causal relationships.

In [ ]:
survey_vars = ['WTINT2YR', 'WTMEC2YR', 'SDMVPSU', 'SDMVSTRA']
present_survey_vars = [v for v in survey_vars if v in model_ready_df.columns]
print(f"[PRESERVED] Survey design variables present in model-ready dataset: {present_survey_vars}")

## Section 12 — Save Processed Outputs

We create the required directories (`data/processed/` and `data/reports/`) and save:
1. `data/processed/nhanes_merged_adults.csv` (Merged adult records with HbA1c audit column)
2. `data/processed/nhanes_model_ready.csv` (Target-labeled records with leakage columns removed)
3. `data/reports/dataset_loading_report.json` (Machine-readable summary)
4. `data/reports/missing_values_report.csv` (Missingness breakdown)
5. `data/reports/class_distribution.csv` (Target class breakdown)
6. `data/reports/column_dictionary.csv` (Column dictionary template)

In [ ]:
processed_dir = Path("data/processed")
reports_dir = Path("data/reports")

processed_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

# 1. Save merged adult dataset (with audit columns)
merged_adults_path = processed_dir / "nhanes_merged_adults.csv"
adults_df.to_csv(merged_adults_path, index=False)

# 2. Save model-ready dataset
model_ready_path = processed_dir / "nhanes_model_ready.csv"
model_ready_df.to_csv(model_ready_path, index=False)

# 3. Save missing values report
missing_report_path = reports_dir / "missing_values_report.csv"
missing_report.to_csv(missing_report_path, index=True)

# 4. Save class distribution report
class_dist_path = reports_dir / "class_distribution.csv"
dist_df.to_csv(class_dist_path, index=True)

# 5. Save column dictionary
col_dict = pd.DataFrame({
    'Column': model_ready_df.columns,
    'DataType': model_ready_df.dtypes.astype(str),
    'Role': ['Target' if c == 'risk_label' else ('Survey Weight' if c in survey_vars else 'Feature') for c in model_ready_df.columns],
    'LeakageStatus': 'Safe'
})
col_dict_path = reports_dir / "column_dictionary.csv"
col_dict.to_csv(col_dict_path, index=False)

# 6. Save JSON loading report
report_data = {
    "project": "CodeCraft - Early Metabolic Risk Detection",
    "source_files": {
        "demographics": len(demographics_df),
        "body_measurements": len(body_measurements_df),
        "glycohemoglobin": len(glycohemoglobin_df)
    },
    "merged_participants_count": len(merged_full),
    "adult_participants_count": len(adults_df),
    "excluded_missing_hba1c": int(missing_ghb),
    "excluded_diabetes_range": int(diabetes_count),
    "final_analysis_sample_size": len(analysis_df),
    "class_distribution": class_counts.to_dict(),
    "class_percentages": class_proportions.to_dict(),
    "num_predictors": int(model_ready_df.shape[1] - 1),
    "survey_variables_retained": present_survey_vars,
    "output_files": {
        "merged_adults": str(merged_adults_path),
        "model_ready": str(model_ready_path),
        "missing_values_report": str(missing_report_path),
        "class_distribution_report": str(class_dist_path),
        "column_dictionary": str(col_dict_path)
    }
}

json_report_path = reports_dir / "dataset_loading_report.json"
with open(json_report_path, "w") as f:
    json.dump(report_data, f, indent=4)

print("[SUCCESS] All processed outputs and reports saved successfully.")

## Section 13 — Final Validation

We execute comprehensive programmatic assertions verifying the integrity of the processed dataset and outputs.

In [ ]:
# 1. Verify files loaded
assert not demographics_df.empty
assert not body_measurements_df.empty
assert not glycohemoglobin_df.empty

# 2. Verify required columns exist
assert 'SEQN' in model_ready_df.columns or 'risk_label' in model_ready_df.columns

# 3. Verify participant IDs are unique in merged files
assert merged_full['SEQN'].is_unique

# 4. Verify target labels belong only to allowed classes
allowed_classes = {'Normal', 'Elevated Risk'}
assert set(model_ready_df['risk_label'].unique()).issubset(allowed_classes)

# 5. Verify no missing target labels
assert model_ready_df['risk_label'].isna().sum() == 0

# 6. Verify no minors remain
assert (model_ready_df['RIDAGEYR'] < 18).sum() == 0

# 7. Verify no leakage columns remain in predictors
assert 'LBXGH' not in model_ready_df.columns
assert 'SEQN' not in model_ready_df.columns

# 8. Verify output files exist and are nonempty
assert model_ready_path.exists() and model_ready_path.stat().st_size > 0
assert json_report_path.exists() and json_report_path.stat().st_size > 0

# 9. Verify reloadability
reloaded_df = pd.read_csv(model_ready_path)
assert len(reloaded_df) == len(model_ready_df)

print("[ALL ASSERTIONS PASSED] Pipeline validation completed successfully without errors.")

In [ ]:
# Final Summary Printout
print("============================================================")
print("       CODECRAFT DATASET LOADING & PREPARATION SUMMARY        ")
print("============================================================")
print(f"• Demographics records loaded:       {len(demographics_df):,}")
print(f"• Body measurements records loaded:  {len(body_measurements_df):,}")
print(f"• Glycohemoglobin records loaded:    {len(glycohemoglobin_df):,}")
print(f"• Merged participants count:         {len(merged_full):,}")
print(f"• Adult participants (Age >= 18):    {len(adults_df):,}")
print(f"• Excluded for missing HbA1c:        {missing_ghb:,}")
print(f"• Excluded for HbA1c >= 6.5%:        {diabetes_count:,}")
print(f"• Final analysis dataset size:       {len(analysis_df):,}")
print("• Target Class Distribution:")
for label, count in class_counts.items():
    pct = class_proportions[label]
    print(f"    - {label}: {count:,} ({pct:.2f}%)")
print(f"• Candidate predictors count:        {model_ready_df.shape[1] - 1}")
print(f"• Model-ready CSV saved to:          {model_ready_path}")
print(f"• Audit report saved to:             {json_report_path}")
print("============================================================")
print("WARNINGS & LIMITATIONS:")
print("1. Cross-sectional survey data captures associations at time of")
print("   survey; causal inferences cannot be established.")
print("2. Complex survey weights must be incorporated in population-level")
print("   evaluations to avoid sample bias.")
print("============================================================")

## Section 14 — Next Steps

The data loading, cleaning, target construction, and validation phase is now complete.  
In the next phase of the CodeCraft project, we will:

1. Review the final feature list and perform feature selection / imputation strategy planning.
2. Create a shared stratified **70/15/15 train-validation-test split** with a fixed random seed (`42`).
3. Fit missing-value imputation, categorical encoding, and feature scaling **strictly on the training partition** to prevent data leakage.
4. Reuse the identical data split across Machine Learning, Deep Learning, and Quantum Machine Learning (QML) comparison pipelines.

*No model training was performed in this notebook.*

# Phase 2 — Train/Validation/Test Split and Preprocessing

In this phase, we build upon the clean, leak-free model-ready dataset produced in Phase 1. We implement:
1. **Feature Pruning & Exclusion:** Removal of survey design variables, audit columns, and pediatric structural variables exhibiting 100% missingness in the adult population.
2. **Stratified Splitting (70/15/15):** Creation of a reproducible, stratified train-validation-test split (seed `42`) ensuring class balance across splits and zero participant overlap.
3. **Isolated Preprocessing Pipeline:** Construction of a `ColumnTransformer` and `Pipeline` for numeric imputation/scaling and categorical imputation/encoding, fitted **strictly on the training partition** to prevent data leakage.
4. **Artifact Persistence:** Saving split datasets, fitted preprocessor (`joblib`), transformation manifests, and summary reports.

## Section 15 — Load Model-Ready Data & Set Random Seed

We load `data/processed/nhanes_model_ready.csv`, verify expected columns and target labels, and establish a reproducible random seed (`42`).

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Establish reproducible seed
SEED = 42
np.random.seed(SEED)

# Load model-ready dataset
model_ready_path = Path("data/processed/nhanes_model_ready.csv")
if not model_ready_path.exists():
    raise FileNotFoundError(f"Model-ready dataset not found at {model_ready_path}. Run Phase 1 first.")

df_model = pd.read_csv(model_ready_path)
print(f"Loaded model-ready dataset shape: {df_model.shape}")

# Verify target column exists
assert 'risk_label' in df_model.columns, "Target column 'risk_label' missing from model-ready dataset."
print(f"Target classes present: {df_model['risk_label'].unique().tolist()}")

## Section 16 — Feature Pruning & Exclusion Strategy

We inspect and exclude:
1. **Target & Identifiers:** `risk_label` (target vector $y$), plus any residual IDs.
2. **Survey Design Variables:** `WTINT2YR`, `WTMEC2YR`, `SDMVPSU`, `SDMVSTRA` (retained for survey-weighted analysis or audit, excluded from ML predictors to prevent weight bias).
3. **Pediatric / 100% Missing Features:** Variables confirmed in Phase 1 to have 100% missingness in the adult population (e.g., `RIDAGEMN`, `BMXRECUM`, `BMIRECUM`, `BMXHEAD`, `BMIHEAD`, `BMIHT`, `BMIARML`, `BMIARMC`, `BMILEG`, etc.).

We document every exclusion and reason.

In [ ]:
# Define exclusion categories with reasons
exclusions = {
    "Target Variable": ["risk_label"],
    "Survey Design Variables": ["WTINT2YR", "WTMEC2YR", "SDMVPSU", "SDMVSTRA"],
    "Pediatric / 100% Missing Features": [
        "RIDAGEMN", "BMXRECUM", "BMIRECUM", "BMXHEAD", "BMIHEAD", 
        "BMIHT", "BMIARML", "BMIARMC", "BMILEG"
    ]
}

# Flatten exclusions list (excluding target from X predictors)
cols_to_drop_from_X = []
for category, cols in exclusions.items():
    for c in cols:
        if c in df_model.columns:
            cols_to_drop_from_X.append(c)

# Separate Features (X) and Target (y)
y = df_model['risk_label'].values
X = df_model.drop(columns=[c for c in cols_to_drop_from_X if c in df_model.columns])

print(f"Feature matrix X shape: {X.shape}")
print(f"Target vector y shape: {y.shape}")
print(f"Excluded columns count: {len(cols_to_drop_from_X)}")
print(f"Excluded columns details:")
for cat, clist in exclusions.items():
    present_clist = [c for c in clist if c in df_model.columns or c == 'risk_label']
    print(f"  - {cat}: {present_clist}")

## Section 17 — Two-Stage Stratified Split (70/15/15)

To prevent data leakage and ensure robust generalization, we perform a two-stage stratified split with seed `42`:
1. **Stage 1:** Split dataset into **Training (70%)** and **Temp (30%)**.
2. **Stage 2:** Split Temp equally into **Validation (15%)** and **Testing (15%)**.

We verify class balance across all splits and confirm zero participant overlap.

In [ ]:
# Stage 1: Train (70%) vs Temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

# Stage 2: Validation (15%) vs Test (15%)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print(f"Training set shape:   {X_train.shape}, Target distribution: {pd.Series(y_train).value_counts().to_dict()}")
print(f"Validation set shape: {X_val.shape}, Target distribution: {pd.Series(y_val).value_counts().to_dict()}")
print(f"Testing set shape:    {X_test.shape}, Target distribution: {pd.Series(y_test).value_counts().to_dict()}")

# Verify split ratios
total_n = len(df_model)
print(f"\nSplit Proportions:")
print(f"  - Train: {len(X_train)} ({len(X_train)/total_n*100:.2f}%)")
print(f"  - Val:   {len(X_val)} ({len(X_val)/total_n*100:.2f}%)")
print(f"  - Test:  {len(X_test)} ({len(X_test)/total_n*100:.2f}%)")

# Verify zero overlap (using DataFrame indices or row positions)
train_idx = set(X_train.index)
val_idx = set(X_val.index)
test_idx = set(X_test.index)

assert train_idx.isdisjoint(val_idx), "Overlap found between train and validation sets!"
assert train_idx.isdisjoint(test_idx), "Overlap found between train and test sets!"
assert val_idx.isdisjoint(test_idx), "Overlap found between validation and test sets!"
print("[VERIFIED] Zero participant overlap across train, validation, and test splits.")

## Section 18 — Isolated Preprocessing Pipeline

We construct a scikit-learn preprocessing pipeline:
- **Numeric Features:** Median imputation (`SimpleImputer`) + Standard Scaling (`StandardScaler`).
- **Categorical Features:** Most-frequent imputation (`SimpleImputer`) + One-Hot Encoding (`OneHotEncoder(handle_unknown='ignore', sparse_output=False)`).

**Critical Rule:** The preprocessor is fitted **strictly on the training data (`X_train`)** and applied to validation and test data without re-fitting.

In [ ]:
# Identify numeric and categorical features
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

print(f"Numeric features count: {len(numeric_features)}")
print(f"Categorical features count: {len(categorical_features)}")

# Build transformers
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
], remainder='drop')

# Fit preprocessor strictly on training data
preprocessor.fit(X_train)

# Transform splits
X_train_transformed = preprocessor.transform(X_train)
X_val_transformed = preprocessor.transform(X_val)
X_test_transformed = preprocessor.transform(X_test)

print(f"Transformed X_train shape: {X_train_transformed.shape}")
print(f"Transformed X_val shape:   {X_val_transformed.shape}")
print(f"Transformed X_test shape:  {X_test_transformed.shape}")

# Verify finite values
assert np.isfinite(X_train_transformed).all(), "Non-finite values found in transformed X_train!"
assert np.isfinite(X_val_transformed).all(), "Non-finite values found in transformed X_val!"
assert np.isfinite(X_test_transformed).all(), "Non-finite values found in transformed X_test!"
print("[VERIFIED] All transformed feature matrices contain finite values only.")

# Extract transformed feature names if possible
try:
    cat_encoder = preprocessor.named_transformers_['cat'].named_steps['onehot']
    encoded_cat_features = cat_encoder.get_feature_names_out(categorical_features).tolist()
    all_transformed_feature_names = numeric_features + encoded_cat_features
except Exception:
    all_transformed_feature_names = [f"feat_{i}" for i in range(X_train_transformed.shape[1])]

print(f"Total transformed feature columns: {len(all_transformed_feature_names)}")

## Section 19 — Save Reproducible Artifacts

We create the required output directories (`data/splits/`, `models/preprocessing/`, `data/reports/`) and save:
1. Split CSVs (`train.csv`, `validation.csv`, `test.csv`) including original features and `risk_label`.
2. Split manifest (`split_manifest.json`)
3. Fitted preprocessor model (`preprocessor.joblib`)
4. Splitting and preprocessing report (`splitting_preprocessing_report.json`)
5. Transformed feature names (`transformed_feature_names.json`)

In [ ]:
splits_dir = Path("data/splits")
models_dir = Path("models/preprocessing")
reports_dir = Path("data/reports")

splits_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

# Reassemble split dataframes (features + target)
train_df = X_train.copy()
train_df['risk_label'] = y_train

val_df = X_val.copy()
val_df['risk_label'] = y_val

test_df = X_test.copy()
test_df['risk_label'] = y_test

# Save CSVs
train_path = splits_dir / "train.csv"
val_path = splits_dir / "validation.csv"
test_path = splits_dir / "test.csv"

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

# Save fitted preprocessor
preprocessor_path = models_dir / "preprocessor.joblib"
joblib.dump(preprocessor, preprocessor_path)

# Save transformed feature names
feature_names_path = reports_dir / "transformed_feature_names.json"
with open(feature_names_path, "w") as f:
    json.dump(all_transformed_feature_names, f, indent=4)

# Save split manifest
manifest = {
    "random_seed": SEED,
    "total_samples": total_n,
    "split_sizes": {
        "train": len(train_df),
        "validation": len(val_df),
        "test": len(test_df)
    },
    "split_proportions": {
        "train": round(len(train_df)/total_n*100, 2),
        "validation": round(len(val_df)/total_n*100, 2),
        "test": round(len(test_df)/total_n*100, 2)
    },
    "class_distributions": {
        "train": pd.Series(y_train).value_counts().to_dict(),
        "validation": pd.Series(y_val).value_counts().to_dict(),
        "test": pd.Series(y_test).value_counts().to_dict()
    },
    "exclusions_applied": exclusions
}

manifest_path = splits_dir / "split_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=4)

# Save splitting and preprocessing report
preprocessing_report = {
    "numeric_features_count": len(numeric_features),
    "categorical_features_count": len(categorical_features),
    "transformed_feature_count": len(all_transformed_feature_names),
    "preprocessor_steps": {
        "numeric": ["median_imputation", "standard_scaler"],
        "categorical": ["most_frequent_imputation", "onehot_encoder"]
    },
    "output_artifacts": {
        "train_csv": str(train_path),
        "validation_csv": str(val_path),
        "test_csv": str(test_path),
        "preprocessor_model": str(preprocessor_path),
        "split_manifest": str(manifest_path),
        "feature_names": str(feature_names_path)
    }
}

report_path = reports_dir / "splitting_preprocessing_report.json"
with open(report_path, "w") as f:
    json.dump(preprocessing_report, f, indent=4)

print("[SUCCESS] All Phase 2 artifacts and reports saved successfully.")

## Section 20 — Validation Checks & Final Summary

We execute comprehensive programmatic assertions verifying the integrity of Phase 2 outputs.

In [ ]:
# 1. Verify split ratios (~70/15/15)
assert abs(len(train_df)/total_n - 0.70) < 0.02
assert abs(len(val_df)/total_n - 0.15) < 0.02
assert abs(len(test_df)/total_n - 0.15) < 0.02

# 2. Verify no missing target labels
assert train_df['risk_label'].isna().sum() == 0
assert val_df['risk_label'].isna().sum() == 0
assert test_df['risk_label'].isna().sum() == 0

# 3. Verify no leakage or survey columns in X_train columns
prohibited = ['SEQN', 'LBXGH', 'WTINT2YR', 'WTMEC2YR', 'SDMVPSU', 'SDMVSTRA']
for p in prohibited:
    assert p not in X_train.columns, f"Prohibited column {p} found in X_train!"

# 4. Verify artifact existence
assert train_path.exists() and train_path.stat().st_size > 0
assert val_path.exists() and val_path.stat().st_size > 0
assert test_path.exists() and test_path.stat().st_size > 0
assert preprocessor_path.exists() and preprocessor_path.stat().st_size > 0
assert manifest_path.exists() and manifest_path.stat().st_size > 0
assert report_path.exists() and report_path.stat().st_size > 0

print("[ALL PHASE 2 ASSERTIONS PASSED] Validation checks completed successfully.")

In [ ]:
# Final Summary Printout
print("============================================================")
print("     CODECRAFT SPLITTING & PREPROCESSING PHASE SUMMARY      ")
print("============================================================")
print(f"• Total analysis participants:       {total_n:,}")
print(f"• Training set size:                 {len(train_df):,} ({len(train_df)/total_n*100:.2f}%)")
print(f"• Validation set size:               {len(val_df):,} ({len(val_df)/total_n*100:.2f}%)")
print(f"• Testing set size:                  {len(test_df):,} ({len(test_df)/total_n*100:.2f}%)")
print(f"• Original feature count:            {X.shape[1]}")
print(f"• Transformed feature count:         {X_train_transformed.shape[1]}")
print(f"• Excluded columns & reasons:")
for cat, cols in exclusions.items():
    print(f"    - {cat}: {cols}")
print(f"• Saved split CSVs path:             {splits_dir}")
print(f"• Saved preprocessor model:          {preprocessor_path}")
print(f"• Saved manifest & report path:      {reports_dir}")
print("============================================================")
print("CRITICAL NOTICE:")
print("The testing set (test.csv) must remain strictly untouched during")
print("all future model selection, hyperparameter tuning, and cross-")
print("validation steps. It will be used only for final evaluation.")
print("============================================================")